# Raster algebra & apply

Run computations over raster values:

- **`apply(func)`** — apply a NumPy function to a band, returning a new `Dataset`.
- **`map_blocks(func, tile_size=...)`** — apply a function tile-by-tile (memory-friendly).
- **`overlay(classes_map)`** — group a raster's values by the classes in a second raster.

(For lazy, Dask-backed algebra see the [Dask quickstart](../dask/dataset.ipynb).)

## Setup

In [ ]:
%matplotlib inline

import tempfile
from pathlib import Path

import numpy as np

DATA = Path('../../../examples/data')
WORK = Path(tempfile.mkdtemp(prefix='pyramids-t3-'))
DATA.is_dir(), WORK.is_dir()

In [ ]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(DATA / 'acc4000.tif')
ds.shape

The source raster — flow accumulation values that the algebra below operates on.

In [ ]:
ds.plot(band=0, title="Source (acc4000)", cmap="viridis")

## Element-wise — `apply`

Apply a function to every cell of a band; returns a new `Dataset`.

In [ ]:
doubled = ds.apply(lambda a: a * 2)
type(doubled).__name__, float(
    np.nanmax(
        np.where(
            doubled.read_array() == doubled.no_data_value[0],
            np.nan,
            doubled.read_array(),
        )
    )
)

The `apply` result — every cell doubled. Same spatial pattern, values scaled by 2.

In [ ]:
doubled.plot(band=0, title="apply: a * 2", cmap="viridis")

## Tile-by-tile — `map_blocks`

Process the raster in `tile_size` blocks — the same result, a bounded memory footprint.

In [ ]:
sen = Dataset.read_file(DATA / 'geotiff' / 'sentinel_crop.tif')
incremented = sen.map_blocks(lambda a: a + 1, tile_size=128)
type(incremented).__name__, incremented.shape

The tile-by-tile `map_blocks` output — the Sentinel crop with every value incremented by 1.

In [ ]:
incremented.plot(band=0, title="map_blocks: a + 1", cmap="viridis")

## Group by class — `overlay`

`overlay` reads a second raster of class labels and returns a dict mapping each class value to
the list of data-raster values that fall in it — a raster-on-raster zonal summary.

In [ ]:
classes = Dataset.read_file(DATA / 'geotiff' / 'sentinel-classes.tif')
grouped = sen.overlay(classes)
type(grouped).__name__, sorted(grouped), [len(v) for v in grouped.values()]

## Notes

- `apply(..., inplace=True)` mutates the dataset; `overlay` takes `exclude_value=` to drop a
  sentinel before grouping.
- See also: [Zonal statistics](zonal-statistics.ipynb) (the vector-polygon equivalent).